# Lorentzian echo side-by-side

Compare the broad echo Lorentzian sweep with its zoomed sweep using the same side-by-side layout as the broad-square spectroscopy plot.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import ConnectionPatch
import numpy as np

from bundle_utils import amp_prefactor_to_rabi_amp_mhz, extract_qubit_variables
from presentation_style import polish_axes, use_presentation_style

use_presentation_style()

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data" / "lorentzian_echo"
FIGURES_DIR = PROJECT_DIR / "figures"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

bundle_paths = sorted(DATA_DIR.glob("*_data_bundle.npz"))
if len(bundle_paths) < 2:
    raise FileNotFoundError(f"Expected at least two data bundles in {DATA_DIR}")

SWEEPS = []
for path in bundle_paths:
    data = extract_qubit_variables(path)
    frequency_span_mhz = data.parameters.get("frequency_span_in_mhz", np.nan)
    peak_amplitude = data.parameters.get("lorentzian_peak_amplitude", 1.0)
    effective_amp = data.amp_prefactor * peak_amplitude
    rabi_amp_mhz = amp_prefactor_to_rabi_amp_mhz(
        effective_amp,
        data.pi_pulse["amplitude"],
        data.pi_pulse["length_ns"],
    )
    SWEEPS.append(
        {
            "path": path,
            "data": data,
            "frequency_span_mhz": frequency_span_mhz,
            "peak_amplitude": peak_amplitude,
            "rabi_amp_mhz": rabi_amp_mhz,
        }
    )

SWEEPS = sorted(SWEEPS, key=lambda sweep: sweep["frequency_span_mhz"], reverse=True)
for index, sweep in enumerate(SWEEPS):
    sweep["label"] = "Broad echo" if index == 0 else "Echo zoom" if index == 1 else f"Sweep {index + 1}"

def set_detuning_ticks(ax, detuning_mhz):
    x_min = float(np.min(detuning_mhz))
    x_max = float(np.max(detuning_mhz))
    if np.isclose(x_min, -50) and np.isclose(x_max, 50):
        ax.set_xticks(np.arange(-50, 51, 10))

metadata = SWEEPS[-1]["data"].metadata
parameters = SWEEPS[-1]["data"].parameters
qubit_name = SWEEPS[-1]["data"].qubit_name
t2_ramsey_ns = SWEEPS[-1]["data"].qubit_profile["transmon"]["t2_ramsey_ns"] / 4
t2_limit_fwhm_mhz = 1000 / (np.pi * t2_ramsey_ns)
t2_limit_half_width_mhz = t2_limit_fwhm_mhz / 2

print(f"Qubit {qubit_name} has T2 limit of {t2_limit_fwhm_mhz:.3f} MHz (from T2 Ramsey = {t2_ramsey_ns:.3f} ns)")
[(sweep["label"], sweep["path"].name, sweep["frequency_span_mhz"]) for sweep in SWEEPS]

In [ ]:
for sweep in SWEEPS:
    data = sweep["data"]
    pi_pulse = data.pi_pulse
    print(f"{sweep['label']}: {sweep['path']}")
    print(f"  Qubit: {data.qubit_name}")
    print(f"  Timestamp: {data.metadata.get('timestamp', 'unknown')}")
    print(f"  Echo: {data.parameters.get('echo', 'unknown')}")
    print(f"  Frequency span: {sweep['frequency_span_mhz']:g} MHz")
    print(f"  Qubit f01: {data.qubit_f01_hz / 1e9:.6f} GHz")
    print(f"  Lorentzian peak amplitude: {sweep['peak_amplitude']:g}")
    print(f"  Pi pulse: {data.pi_pulse_name}, amp={pi_pulse['amplitude']:.9f}, length={pi_pulse['length_ns']} ns")
    print(f"  Rabi scale: {amp_prefactor_to_rabi_amp_mhz(1, pi_pulse['amplitude'], pi_pulse['length_ns']):.3f} MHz per amplitude unit")
    print(f"  Result shape: {data.result.shape} = detuning x amp_prefactor")
    print(f"  Detuning range: {np.min(data.detuning_hz / 1e6):g} to {np.max(data.detuning_hz / 1e6):g} MHz")
    print(f"  Effective Rabi amplitude range: {np.min(sweep['rabi_amp_mhz']):g} to {np.max(sweep['rabi_amp_mhz']):g} MHz")
print(f"T2 Ramsey: {t2_ramsey_ns:g} ns")
print(f"T2-limited FWHM: {t2_limit_fwhm_mhz:.5f} MHz")

In [ ]:
vmin = min(float(np.nanmin(sweep["data"].result)) for sweep in SWEEPS)
vmax = max(float(np.nanmax(sweep["data"].result)) for sweep in SWEEPS)

sweep = SWEEPS[0]
data = sweep["data"]
detuning_mhz = data.detuning_hz / 1e6

fig = plt.figure(figsize=(12.0, 4.8))
axes = [
    fig.add_axes([0.07, 0.16, 0.39, 0.78]),
    fig.add_axes([0.53, 0.16, 0.39, 0.78]),
]
ax = axes[0]
axes[1].set_axis_off()
cax = fig.add_axes([0.955, 0.18, 0.018, 0.74])
mesh = ax.pcolormesh(
    detuning_mhz,
    sweep["rabi_amp_mhz"],
    data.result.T,
    shading="auto",
    cmap="viridis",
    vmin=vmin,
    vmax=vmax,
)
ax.text(
    0.03,
    0.95,
    sweep["label"],
    transform=ax.transAxes,
    va="top",
    ha="left",
    fontsize=12,
    color="white",
    bbox={"facecolor": "black", "alpha": 0.35, "edgecolor": "none", "pad": 4},
)
ax.set_xlabel(r"Drive detuning $(f_d-f_{01})$ (MHz)")
ax.set_ylabel("Effective Rabi amplitude (MHz)")
ax.set_xlim(float(np.min(detuning_mhz)), float(np.max(detuning_mhz)))
set_detuning_ticks(ax, detuning_mhz)
ax.set_ylim(float(np.min(sweep["rabi_amp_mhz"])), float(np.max(sweep["rabi_amp_mhz"])))
polish_axes(ax)

cbar = fig.colorbar(mesh, cax=cax)
cbar.set_label("Excitation", fontsize=12)
cbar.outline.set_visible(False)

figure_stem = FIGURES_DIR / f"04_lorentzian_echo_{qubit_name}_broad_only"
png_path = figure_stem.with_suffix(".png")
pdf_path = figure_stem.with_suffix(".pdf")
fig.canvas.draw()
fig.savefig(png_path, dpi=300, bbox_inches=None, facecolor="white")
fig.savefig(pdf_path, bbox_inches=None, facecolor="white")

plt.show()

print(f"Saved PNG: {png_path.resolve()}")
print(f"Saved PDF: {pdf_path.resolve()}")

In [ ]:
vmin = min(float(np.nanmin(sweep["data"].result)) for sweep in SWEEPS)
vmax = max(float(np.nanmax(sweep["data"].result)) for sweep in SWEEPS)

fig = plt.figure(figsize=(12.0, 4.8))
axes = [
    fig.add_axes([0.07, 0.16, 0.39, 0.78]),
    fig.add_axes([0.53, 0.16, 0.39, 0.78]),
]
cax = fig.add_axes([0.955, 0.18, 0.018, 0.74])

mesh = None
for index, (ax, sweep) in enumerate(zip(axes, SWEEPS)):
    data = sweep["data"]
    detuning_mhz = data.detuning_hz / 1e6
    mesh = ax.pcolormesh(
        detuning_mhz,
        sweep["rabi_amp_mhz"],
        data.result.T,
        shading="auto",
        cmap="viridis",
        vmin=vmin,
        vmax=vmax,
    )
    ax.text(
        0.03,
        0.95,
        sweep["label"],
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=12,
        color="white",
        bbox={"facecolor": "black", "alpha": 0.35, "edgecolor": "none", "pad": 4},
    )
    if index == 1:
        ax.axvline(-t2_limit_half_width_mhz, color="white", linestyle="--", linewidth=1.1, alpha=0.9, zorder=6)
        ax.axvline(t2_limit_half_width_mhz, color="white", linestyle="--", linewidth=1.1, alpha=0.9, zorder=6)
        ax.axvspan(-t2_limit_half_width_mhz, t2_limit_half_width_mhz, color="white", alpha=0.08, zorder=4)
    ax.set_xlabel(r"Drive detuning $(f_d-f_{01})$ (MHz)")
    ax.set_ylabel("Effective Rabi amplitude (MHz)")
    ax.set_xlim(float(np.min(detuning_mhz)), float(np.max(detuning_mhz)))
    set_detuning_ticks(ax, detuning_mhz)
    ax.set_ylim(float(np.min(sweep["rabi_amp_mhz"])), float(np.max(sweep["rabi_amp_mhz"])))
    polish_axes(ax)

big_sweep = SWEEPS[0]
zoom_sweep = SWEEPS[1]
zoom_x_min = float(np.min(zoom_sweep["data"].detuning_hz / 1e6))
zoom_x_max = float(np.max(zoom_sweep["data"].detuning_hz / 1e6))
zoom_y_min = float(np.min(zoom_sweep["rabi_amp_mhz"]))
zoom_y_max = float(np.max(zoom_sweep["rabi_amp_mhz"]))
axes[0].add_patch(
    plt.Rectangle(
        (zoom_x_min, zoom_y_min),
        zoom_x_max - zoom_x_min,
        zoom_y_max - zoom_y_min,
        fill=False,
        edgecolor="#d62728",
        linewidth=2.0,
        zorder=6,
    )
)

for rect_y, zoom_y in [(zoom_y_min, zoom_y_min), (zoom_y_max, zoom_y_max)]:
    connector = ConnectionPatch(
        xyA=(zoom_x_max, rect_y),
        coordsA=axes[0].transData,
        xyB=(zoom_x_min, zoom_y),
        coordsB=axes[1].transData,
        color="#d62728",
        linewidth=1.4,
        alpha=0.9,
        zorder=7,
        clip_on=False,
    )
    fig.add_artist(connector)

cbar = fig.colorbar(mesh, cax=cax)
cbar.set_label("Excitation", fontsize=12)
cbar.outline.set_visible(False)

figure_stem = FIGURES_DIR / f"04_lorentzian_echo_{qubit_name}_two_sweeps"
png_path = figure_stem.with_suffix(".png")
pdf_path = figure_stem.with_suffix(".pdf")
# fig.canvas.draw()
fig.savefig(png_path, dpi=300, bbox_inches=None, facecolor="white")
fig.savefig(pdf_path, bbox_inches=None, facecolor="white")

plt.show()

print(f"Saved PNG: {png_path.resolve()}")
print(f"Saved PDF: {pdf_path.resolve()}")

In [ ]:
target_rabi_amp_mhz = np.array([0.15, 0.30, 0.50, 0.75]) * np.nanmax(SWEEPS[-1]["rabi_amp_mhz"])

fig, axes = plt.subplots(1, len(SWEEPS), figsize=(12.0, 4.8), sharey=True, constrained_layout=True)
if len(SWEEPS) == 1:
    axes = [axes]

for ax, sweep in zip(axes, SWEEPS):
    data = sweep["data"]
    detuning_mhz = data.detuning_hz / 1e6
    rabi_amp_mhz = sweep["rabi_amp_mhz"]
    slice_indices = sorted({int(np.nanargmin(np.abs(rabi_amp_mhz - target))) for target in target_rabi_amp_mhz})

    for slice_index in slice_indices:
        ax.plot(
            detuning_mhz,
            data.result[:, slice_index],
            linewidth=2.0,
            label=f"{rabi_amp_mhz[slice_index]:.2f} MHz",
        )

    ax.axvline(-t2_limit_half_width_mhz, color="#666666", linestyle="--", linewidth=1.0, alpha=0.75)
    ax.axvline(t2_limit_half_width_mhz, color="#666666", linestyle="--", linewidth=1.0, alpha=0.75)
    ax.set_title(sweep["label"], fontsize=13, pad=8)
    ax.set_xlabel(r"Drive detuning $(f_d-f_{01})$ (MHz)")
    ax.set_xlim(float(np.min(detuning_mhz)), float(np.max(detuning_mhz)))
    set_detuning_ticks(ax, detuning_mhz)
    polish_axes(ax)

axes[0].set_ylabel("Excitation")
axes[-1].legend(title="Rabi amplitude", loc="best", fontsize=10, title_fontsize=10)

figure_stem = FIGURES_DIR / f"04_lorentzian_echo_{qubit_name}_slices"
png_path = figure_stem.with_suffix(".png")
pdf_path = figure_stem.with_suffix(".pdf")
fig.savefig(png_path, dpi=300, facecolor="white")
fig.savefig(pdf_path, facecolor="white")

plt.show()

print(f"Saved PNG: {png_path.resolve()}")
print(f"Saved PDF: {pdf_path.resolve()}")